In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
from typing import Tuple, List
from datetime import datetime
import time
import warnings
import io
import csv
import requests
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Suppress all FutureWarning messages
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==============================================================================
# 1. LOAD DATA & CALCULATE SPATIAL DIFFERENCES
# ==============================================================================
df = pd.read_csv('Result 2025-09-23 05-01-51.csv')

df['focus_asset_id'] = df['focus_asset_id'].astype('Int64')
df['focus_asset'] = df['focus_asset'].astype('string')
df['associated_asset_id'] = df['associated_asset_id'].astype('Int64')
df['source'] = df['source'].astype('string')
df['name'] = df['name'].astype('string')
df['operator_site_id'] = df['operator_site_id'].astype('string')
df['type'] = df['type'].astype('string')
df['description'] = df['description'].astype('string')
df['operator_name'] = df['operator_name'].astype('string')
df['manager_name'] = df['manager_name'].astype('string')
df['fcc_owner_name'] = df['fcc_owner_name'].astype('string')
df['shelter'] = df['shelter'].astype('string')
df['power'] = df['power'].astype('string')
df['fcc_asr_number'] = df['fcc_asr_number'].astype('string')
df['faa_study_number'] = df['faa_study_number'].astype('string')
df['cdbs_facility_id'] = df['cdbs_facility_id'].astype('string')
df['region'] = df['region'].astype('string')
df['address'] = df['address'].astype('string')
df['stealth'] = df['stealth'].astype('string')
df['asset_status'] = df['asset_status'].astype('string')
df['audit_reason'] = df['audit_reason'].astype('string')

In [2]:
def calculate_distances_to_reference(df):
    df_copy = df.copy()
    df_copy['distance_to_reference'] = np.nan
    for group_name, group_df in df_copy.groupby('focus_asset_id'):
        reference_record = group_df[group_df['associated_asset_id'].isna()]
        if not reference_record.empty:
            ref_lat = reference_record['latitude'].iloc[0]
            ref_lon = reference_record['longitude'].iloc[0]
            reference_coords = (ref_lat, ref_lon)
            for index, row in group_df.iterrows():
                if pd.notna(row['associated_asset_id']):
                    record_coords = (row['latitude'], row['longitude'])
                    distance = geodesic(reference_coords, record_coords).meters
                    df_copy.loc[index, 'distance_to_reference'] = distance
    return df_copy

df_with_distances = calculate_distances_to_reference(df)

In [3]:
def calculate_agldiff_to_reference(df):
    df_copy = df_with_distances.copy()
    df_copy['agldiff_to_reference'] = np.nan
    for group_name, group_df in df_copy.groupby('focus_asset_id'):
        reference_record = group_df[group_df['associated_asset_id'].isna()]
        if not reference_record.empty:
            ref_agl = reference_record['agl'].iloc[0]
            for index, row in group_df.iterrows():
                if pd.notna(row['associated_asset_id']):
                    record_agl = row['agl']
                    distance = ref_agl - record_agl
                    df_copy.loc[index, 'agldiff_to_reference'] = distance
    return df_copy

prox_audits_table = calculate_agldiff_to_reference(df_with_distances)

In [4]:
# ==============================================================================
# 2. BULK DATA EXTRACTION (Replaces row-by-row live API requests)
# ==============================================================================
REGIONS = ['AAL', 'ACE', 'AEA', 'AGL', 'ANE', 'ANM', 'ASO', 'ASW', 'AWP', 'WTE', 'WTW']
START_YEAR = 1960
END_YEAR = 2026
BASE_URL = "https://oeaaa.faa.gov/oeaaa/oe3a-external-api/downloadArchives.do"
TARGET_COLS = ['STUDY (ASN)', 'ENTERED DATE', 'FCC NUMBER']
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Referer': 'https://oeaaa.faa.gov/oeaaa/oe3a/main/'
}

def fetch_region_year(region, year):
    fname = f"Part77{region}{year}List.gzip"
    url = f"{BASE_URL}?fname={fname}"
    for attempt in range(2):
        try:
            response = requests.get(url, headers=HEADERS, timeout=8)
            if response.status_code == 200 and len(response.content) > 100:
                f = io.StringIO(response.text)
                reader = csv.reader(f)
                header = [col.strip() for col in next(reader, [])]
                col_indices = {name: header.index(name) for name in TARGET_COLS if name in header}
                if not col_indices:
                    return (False, None)
                max_idx = max(col_indices.values())
                extracted_data = []
                for row in reader:
                    if len(row) > max_idx:
                        extracted_data.append({name: row[idx].strip() for name, idx in col_indices.items()})
                if extracted_data:
                    return (True, pd.DataFrame(extracted_data))
            elif response.status_code == 404:
                return (False, None)
        except Exception:
            if attempt == 1:
                return (False, None)
            time.sleep(0.5)
    return (False, None)

print("\n🚀 Downloading FAA Part 77 archives to build local lookup dictionaries...", flush=True)
tasks = [(region, year) for year in range(END_YEAR, START_YEAR - 1, -1) for region in REGIONS]
all_dfs = []

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = [executor.submit(fetch_region_year, r, y) for r, y in tasks]
    for future in tqdm(as_completed(futures), total=len(tasks), desc="Fetching FAA Archives"):
        success, df_temp = future.result()
        if success and df_temp is not None:
            all_dfs.append(df_temp)

# Global memory caches
asn_cache = {}
asr_cache = {}

if all_dfs:
    master_df = pd.concat(all_dfs, ignore_index=True)
    master_df['FCC NUMBER'] = master_df['FCC NUMBER'].astype(str).str.strip()
    master_df['STUDY (ASN)'] = master_df['STUDY (ASN)'].astype(str).str.strip()
    master_df['ENTERED DATE_DT'] = pd.to_datetime(master_df['ENTERED DATE'], errors='coerce')

    # Sort dataset newest first to ensure we cache the most recent record
    master_df = master_df.sort_values(by=['ENTERED DATE_DT'], ascending=[False])

    # Drop rows missing FCC or ASN for dictionary building
    valid_fcc_df = master_df[(master_df['FCC NUMBER'] != '') & (master_df['FCC NUMBER'] != 'nan')].drop_duplicates(subset=['FCC NUMBER'], keep='first')
    valid_asn_df = master_df[(master_df['STUDY (ASN)'] != '') & (master_df['STUDY (ASN)'] != 'nan')].drop_duplicates(subset=['STUDY (ASN)'], keep='first')

    # Build instantaneous dictionaries
    asn_cache = dict(zip(valid_fcc_df['FCC NUMBER'], valid_fcc_df['STUDY (ASN)']))
    asr_cache = dict(zip(valid_asn_df['STUDY (ASN)'], valid_asn_df['FCC NUMBER']))

    print(f"✅ Memory caches built! {len(asn_cache):,} ASR keys and {len(asr_cache):,} ASN keys ready for instant lookup.", flush=True)
else:
    print("⚠️ Warning: No FAA data was extracted. Fallback methods will be used.", flush=True)


def parse_asn(asn_string):
    match = re.match(r"(\d{4})-([A-Z]{3})-([\d\w]+)-([A-Z]{2})", str(asn_string))
    if match:
        year, region, sequence, casetype = match.groups()
        return {
            "asnYear": int(year),
            "asnRegion": region,
            "asnSequence": sequence,
            "asnCaseType": casetype
        }
    return None

# Instantaneous Local Dictionary Lookups
def get_asn_via_api(asr_number):
    return asn_cache.get(str(asr_number).strip(), None)

def get_asr_via_api(asn_number):
    if not parse_asn(str(asn_number).strip()):
        return None
    return asr_cache.get(str(asn_number).strip(), None)


# ==============================================================================
# 3. CORE LOGIC (Untouched: determining final numbers, cleaning, cases 1-5)
# ==============================================================================

def determine_faa_study_number(ref_fcc, ref_faa, assoc_faa):
    new_faa = get_asn_via_api(ref_fcc)
    if pd.notnull(new_faa) and new_faa != 'N/A':
        return str(new_faa)
    faa_ids_to_check = set()
    if pd.notnull(ref_faa) and parse_asn(str(ref_faa)): faa_ids_to_check.add(str(ref_faa))
    if pd.notnull(assoc_faa) and parse_asn(str(assoc_faa)): faa_ids_to_check.add(str(assoc_faa))
    for faa_id in faa_ids_to_check:
        found_asr = get_asr_via_api(faa_id)
        if pd.notnull(found_asr) and str(found_asr) == str(ref_fcc):
            return faa_id
    return None

def get_case_1_final_faa(ref_fcc, ref_faa, assoc_faa):
    new_faa = get_asn_via_api(ref_fcc)
    if pd.notnull(new_faa) and new_faa != 'N/A':
        return str(new_faa)
    else:
        return ref_faa

def clean_asr_in_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    if 'fcc_asr_number' in df.columns:
        original_strings = df['fcc_asr_number'].copy()
        numeric_fcc = pd.to_numeric(df['fcc_asr_number'], errors='coerce')
        def to_clean_str(x, original_val):
            if pd.notnull(x) and x == int(x): return str(int(x))
            if pd.notnull(original_val): return str(original_val)
            return np.nan
        df['fcc_asr_number'] = [to_clean_str(num, orig) for num, orig in zip(numeric_fcc, original_strings)]
    return df



🚀 Downloading FAA Part 77 archives to build local lookup dictionaries...


Fetching FAA Archives: 100%|██████████| 737/737 [01:07<00:00, 10.99it/s]


✅ Memory caches built! 155,168 ASR keys and 1,795,706 ASN keys ready for instant lookup.


In [5]:
def merge_records(reference_record, associated_record, merge_timestamp, faa_study_number=None):
    merged_record = reference_record.copy()
    ref_op = reference_record['operator_name']
    assoc_op = associated_record['operator_name']

    if pd.notnull(ref_op) and ref_op in ["Unassigned", "Unkown"] and \
       pd.notnull(assoc_op) and assoc_op not in ["Unassigned", "Unkown"]:
        merged_record['operator_name'] = assoc_op

    merged_record['associated_asset_id'] = reference_record['associated_asset_id']
    merged_record['source'] = f"Auto-Merged {merge_timestamp.strftime('%m/%Y')}"
    merged_record['created_at'] = reference_record['created_at']
    merged_record['updated_at'] = merge_timestamp.strftime('%Y-%m-%d %H:%M:%S')

    fields_to_check = [
        "latitude", "longitude", "name", "operator_site_id", "type", "description",
        "manager_name", "fcc_owner_name", "agl", "amsl", "ground_elevation", "haat",
        "shelter", "power", "stories", "fcc_asr_number", "cdbs_facility_id", "region",
        "address", "construction_date", "stealth", "asset_status"
    ]
    for field in fields_to_check:
        if pd.isnull(merged_record[field]) and pd.notnull(associated_record[field]):
            merged_record[field] = associated_record[field]
    if faa_study_number is not None:
        merged_record['faa_study_number'] = faa_study_number

    if pd.notnull(merged_record['fcc_asr_number']):
        try:
            numeric_val = pd.to_numeric(merged_record['fcc_asr_number'], errors='coerce')
            if pd.notnull(numeric_val) and numeric_val == int(numeric_val):
                merged_record['fcc_asr_number'] = str(int(numeric_val))
            else:
                merged_record['fcc_asr_number'] = str(merged_record['fcc_asr_number'])
        except Exception:
            pass

    if pd.notnull(merged_record['construction_date']):
        try:
            pd.to_datetime(merged_record['construction_date'])
            merged_record['asset_status'] = "Active"
        except:
            pass
    return merged_record

# ----------------- CASE 1 -----------------
def split_case_1_audits(prox_audits_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = prox_audits_table.copy()
    working_table = clean_asr_in_dataframe(working_table)
    candidates_indices = set()
    grouped = working_table.groupby('focus_asset_id')
    for focus_asset_id, group in grouped:
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None

        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            ref_fcc = reference_record['fcc_asr_number']
            ref_faa = reference_record['faa_study_number']
            if pd.notnull(ref_fcc) and pd.notnull(ref_faa):
                matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
                                (associated_records_df['faa_study_number'] == ref_faa)
                matching_associated_records = associated_records_df[matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)

    case1_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    all_original_indices = set(working_table.index)
    final_post_merge_indices = all_original_indices.difference(candidates_indices)
    case1_prox_audits_post_auto_merge_table = working_table.loc[list(final_post_merge_indices)].copy()

    cols = working_table.columns
    if case1_auto_merge_candidates.empty:
         case1_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case1_prox_audits_post_auto_merge_table.empty:
         case1_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case1_auto_merge_candidates, case1_prox_audits_post_auto_merge_table

def apply_case_1_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    start_time = time.time()
    print("Starting Case 1 processing...")
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    case1_auto_merge_further_filter_list = []
    case1_raw_post_auto_merge_list = []
    case1_post_auto_merge_list = []
    new_prox_audit_records_list = []
    failed_prox_audit_records_list = []
    case1_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    processed_focus_ids = set()
    processed_assoc_ids = set()

    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')
    grouped = candidates_table.groupby('focus_asset_id')

    for focus_asset_id, group in tqdm(grouped, desc="Processing Case 1"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        if len(reference_record_df) != 1: continue
        reference_record = reference_record_df.iloc[0]
        ref_fcc = reference_record['fcc_asr_number']
        ref_faa = reference_record['faa_study_number']
        ref_source = reference_record['source']
        ref_year_month = reference_record['created_at_month']
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        if processed_focus_ids:
            focus_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)
            indices_to_remove.update(remaining_associated_records_df[focus_id_match_mask].index)
        if processed_assoc_ids:
            assoc_id_match_mask = remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)
            indices_to_remove.update(remaining_associated_records_df[assoc_id_match_mask].index)

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        date_source_match_mask = (remaining_associated_records_df['source'] == ref_source) & \
                                 (remaining_associated_records_df['created_at_month'] == ref_year_month)
        indices_to_remove.update(remaining_associated_records_df[date_source_match_mask].index)

        removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)]

        for df_target in [removed_associated_records_df, remaining_associated_records_df]:
            if 'created_at_month' in df_target.columns: df_target.drop(columns=['created_at_month'], inplace=True)

        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)

        if not removed_associated_records_df.empty:
            failed_prox_audit_records_list.append(removed_associated_records_df)

        if len(remaining_associated_records_df) > 0:
            current_group_to_merge = pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False)
            case1_auto_merge_further_filter_list.append(current_group_to_merge)
            for index, assoc_record in remaining_associated_records_df.iterrows():
                final_faa = get_case_1_final_faa(ref_fcc, ref_faa, assoc_record['faa_study_number'])
                merged_record_series = merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)
                merged_record_df = pd.DataFrame([merged_record_series], columns=cols)
                new_prox_audit_records_list.append(merged_record_df)
                case1_post_auto_merge_list.append(merged_record_df)
                case1_raw_post_auto_merge_list.append(merged_record_df)
                assoc_record_df = pd.DataFrame([assoc_record], columns=cols)
                case1_raw_post_auto_merge_list.append(assoc_record_df)
                ref_record_df_cleaned_raw = ref_record_cleaned.copy()
                case1_raw_post_auto_merge_list.append(ref_record_df_cleaned_raw)
                if pd.notnull(merged_record_series['focus_asset_id']):
                    processed_focus_ids.add(merged_record_series['focus_asset_id'])
                if pd.notnull(assoc_record['associated_asset_id']):
                    processed_assoc_ids.add(assoc_record['associated_asset_id'])
        else:
            failed_prox_audit_records_list.append(ref_record_cleaned)

    if new_prox_audit_records_list:
        new_records_df = pd.concat(new_prox_audit_records_list, ignore_index=True)
        case1_prox_audits_post_auto_merge_table = pd.concat([case1_prox_audits_post_auto_merge_table, new_records_df], ignore_index=True)
    if failed_prox_audit_records_list:
        failed_records_df = pd.concat(failed_prox_audit_records_list, ignore_index=True)
        case1_prox_audits_post_auto_merge_table = pd.concat([case1_prox_audits_post_auto_merge_table, failed_records_df], ignore_index=True)

    case1_auto_merge_further_filter = pd.concat(case1_auto_merge_further_filter_list, ignore_index=False) if case1_auto_merge_further_filter_list else pd.DataFrame(columns=cols)
    case1_post_auto_merge_table = pd.concat(case1_post_auto_merge_list, ignore_index=True) if case1_post_auto_merge_list else pd.DataFrame(columns=cols)
    case1_raw_post_auto_merge_table = pd.concat(case1_raw_post_auto_merge_list, ignore_index=True) if case1_raw_post_auto_merge_list else pd.DataFrame(columns=cols)
    print(f"--- Case 1 Processing completed in: {(time.time() - start_time) / 60:.2f} minutes ---")

    return (case1_auto_merge_further_filter, case1_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
            case1_post_auto_merge_table.drop_duplicates(ignore_index=True), case1_raw_post_auto_merge_table.drop_duplicates(ignore_index=True))

def apply_case_1_maintenance_logic(prox_audits_table: pd.DataFrame, post_auto_merge_table: pd.DataFrame, post_merge_table: pd.DataFrame, raw_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if post_auto_merge_table.empty: return pd.DataFrame(columns=prox_audits_table.columns), pd.DataFrame(columns=prox_audits_table.columns)
    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    associated_records_mask = working_post_merge['associated_asset_id'].notnull()
    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        cols_to_update = [col for col in working_post_merge.columns.tolist() if col not in ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]]
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            if assoc_record['associated_asset_id'] in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_record['associated_asset_id']]
                if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index and col in working_post_merge.columns:
                        working_post_merge.loc[idx, col] = matching_merged_record[col]
    if not raw_post_merge_table.empty:
        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]
    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_list.append(working_post_merge[working_post_merge['focus_asset_id'].isin(single_record_groups)].copy())
        working_post_merge = working_post_merge[~working_post_merge['focus_asset_id'].isin(single_record_groups)]
    sorted_post_merge_table = working_post_merge.sort_values(by=['focus_asset_id', 'associated_asset_id'], ascending=[True, True], na_position='first').reset_index(drop=True)
    final_asset_table = pd.concat(final_asset_table_list, ignore_index=True)
    if final_asset_table.empty: final_asset_table = pd.DataFrame(columns=prox_audits_table.columns)
    return final_asset_table, sorted_post_merge_table


# ----------------- CASE 2 -----------------
def split_case_2_audits(case1_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = clean_asr_in_dataframe(case1_sorted_post_merge_table.copy())
    candidates_indices = set()
    for focus_asset_id, group in working_table.groupby('focus_asset_id'):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        ref_index = reference_record_df.index[0] if len(reference_record_df) == 1 else None
        if len(reference_record_df) == 1 and ref_index is not None:
            reference_record = reference_record_df.iloc[0]
            ref_fcc, ref_faa = reference_record['fcc_asr_number'], reference_record['faa_study_number']
            if pd.notnull(ref_fcc) and pd.notnull(ref_faa):
                matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & \
                                (associated_records_df['faa_study_number'] != ref_faa) & \
                                (associated_records_df['fcc_asr_number'].notnull()) & \
                                (associated_records_df['faa_study_number'].notnull())
                matching_associated_records = associated_records_df[matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.add(ref_index)
                    candidates_indices.update(matching_associated_records.index)
    case2_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    case2_prox_audits_post_auto_merge_table = working_table.loc[list(set(working_table.index).difference(candidates_indices))].copy()
    cols = working_table.columns
    if case2_auto_merge_candidates.empty: case2_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case2_prox_audits_post_auto_merge_table.empty: case2_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case2_auto_merge_candidates, case2_prox_audits_post_auto_merge_table

def apply_case_2_full_processing(candidates_table: pd.DataFrame, initial_prox_audits_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    start_time = time.time()
    print("Starting Case 2 processing...")
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    case2_auto_merge_further_filter_list, case2_raw_post_auto_merge_list, case2_post_auto_merge_list = [], [], []
    new_prox_audit_records_list, failed_prox_audit_records_list = [], []
    case2_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    processed_focus_ids, processed_assoc_ids = set(), set()

    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')

    for focus_asset_id, group in tqdm(candidates_table.groupby('focus_asset_id'), desc="Processing Case 2"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        if len(reference_record_df) != 1: continue
        reference_record = reference_record_df.iloc[0]
        ref_fcc, ref_faa, ref_source, ref_year_month = reference_record['fcc_asr_number'], reference_record['faa_study_number'], reference_record['source'], reference_record['created_at_month']
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        if processed_focus_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)].index)
        if processed_assoc_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)].index)
        indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['source'] == merge_source_check].index)

        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        if reference_record['source'] == merge_source_check:
            if 'created_at_month' in associated_records_df.columns: associated_records_df.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not associated_records_df.empty: failed_prox_audit_records_list.append(associated_records_df.copy())
            continue

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        indices_to_remove.update(remaining_associated_records_df[(remaining_associated_records_df['source'] == ref_source) & (remaining_associated_records_df['created_at_month'] == ref_year_month)].index)
        removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)]

        for df_target in [removed_associated_records_df, remaining_associated_records_df]:
            if 'created_at_month' in df_target.columns: df_target.drop(columns=['created_at_month'], inplace=True)
        if not removed_associated_records_df.empty: failed_prox_audit_records_list.append(removed_associated_records_df)

        if len(remaining_associated_records_df) > 0:
            case2_auto_merge_further_filter_list.append(pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False))
            for index, assoc_record in remaining_associated_records_df.iterrows():
                final_faa = determine_faa_study_number(ref_fcc, ref_faa, assoc_record['faa_study_number'])
                if pd.notnull(final_faa):
                    merged_record_df = pd.DataFrame([merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)], columns=cols)
                    new_prox_audit_records_list.append(merged_record_df)
                    case2_post_auto_merge_list.append(merged_record_df)
                    case2_raw_post_auto_merge_list.extend([merged_record_df, pd.DataFrame([assoc_record], columns=cols), ref_record_cleaned.copy()])
                    if pd.notnull(merged_record_df['focus_asset_id'].iloc[0]): processed_focus_ids.add(merged_record_df['focus_asset_id'].iloc[0])
                    if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
                else:
                    failed_prox_audit_records_list.extend([ref_record_cleaned, pd.DataFrame([assoc_record], columns=cols)])
        else: failed_prox_audit_records_list.append(ref_record_cleaned)

    if new_prox_audit_records_list: case2_prox_audits_post_auto_merge_table = pd.concat([case2_prox_audits_post_auto_merge_table] + new_prox_audit_records_list, ignore_index=True)
    if failed_prox_audit_records_list: case2_prox_audits_post_auto_merge_table = pd.concat([case2_prox_audits_post_auto_merge_table] + failed_prox_audit_records_list, ignore_index=True)

    case2_auto_merge_further_filter = pd.concat(case2_auto_merge_further_filter_list, ignore_index=False) if case2_auto_merge_further_filter_list else pd.DataFrame(columns=cols)
    case2_post_auto_merge_table = pd.concat(case2_post_auto_merge_list, ignore_index=True) if case2_post_auto_merge_list else pd.DataFrame(columns=cols)
    case2_raw_post_auto_merge_table = pd.concat(case2_raw_post_auto_merge_list, ignore_index=True) if case2_raw_post_auto_merge_list else pd.DataFrame(columns=cols)
    print(f"--- Case 2 Processing completed in: {(time.time() - start_time) / 60:.2f} minutes ---")
    return (case2_auto_merge_further_filter, case2_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True), case2_post_auto_merge_table.drop_duplicates(ignore_index=True), case2_raw_post_auto_merge_table.drop_duplicates(ignore_index=True))

def apply_case_2_maintenance_logic(prox_audits_table, post_auto_merge_table, post_merge_table, raw_post_merge_table, running_final_asset_table):
    if post_auto_merge_table.empty: return running_final_asset_table, pd.DataFrame(columns=prox_audits_table.columns)
    final_asset, sorted_post_merge = apply_case_1_maintenance_logic(prox_audits_table, post_auto_merge_table, post_merge_table, raw_post_merge_table)
    return pd.concat([running_final_asset_table, final_asset], ignore_index=True), sorted_post_merge


# ----------------- CASE 3 -----------------
def split_case_3_audits(case2_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = clean_asr_in_dataframe(case2_sorted_post_merge_table.copy())
    candidates_indices = set()
    for focus_asset_id, group in working_table.groupby('focus_asset_id'):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        if len(reference_record_df) == 1:
            reference_record = reference_record_df.iloc[0]
            ref_fcc, ref_faa = reference_record['fcc_asr_number'], reference_record['faa_study_number']
            if pd.notnull(ref_fcc):
                mask_ref_null, mask_assoc_not_null = pd.isnull(ref_faa), associated_records_df['faa_study_number'].notnull()
                mask_ref_not_null, mask_assoc_null = pd.notnull(ref_faa), associated_records_df['faa_study_number'].isnull()
                matching_mask = (associated_records_df['fcc_asr_number'] == ref_fcc) & (associated_records_df['fcc_asr_number'].notnull()) & ((mask_ref_null & mask_assoc_not_null) | (mask_ref_not_null & mask_assoc_null))
                matching_associated_records = associated_records_df[matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.update([reference_record_df.index[0]] + list(matching_associated_records.index))
    case3_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    case3_prox_audits_post_auto_merge_table = working_table.loc[list(set(working_table.index).difference(candidates_indices))].copy()
    cols = working_table.columns
    if case3_auto_merge_candidates.empty: case3_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case3_prox_audits_post_auto_merge_table.empty: case3_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case3_auto_merge_candidates, case3_prox_audits_post_auto_merge_table

def apply_case_3_full_processing(
    candidates_table: pd.DataFrame,
    initial_prox_audits_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    start_time = time.time()
    print("Starting Case 3 processing...")
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"

    case3_auto_merge_further_filter_list, case3_raw_post_auto_merge_list, case3_post_auto_merge_list = [], [], []
    new_prox_audit_records_list, failed_prox_audit_records_list = [], []
    case3_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    processed_focus_ids, processed_assoc_ids = set(), set()

    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']):
        candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')

    for focus_asset_id, group in tqdm(candidates_table.groupby('focus_asset_id'), desc="Processing Case 3"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        if len(reference_record_df) != 1: continue
        reference_record = reference_record_df.iloc[0]
        ref_fcc, ref_faa = reference_record['fcc_asr_number'], reference_record['faa_study_number']
        ref_source, ref_year_month = reference_record['source'], reference_record['created_at_month']

        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()

        if processed_focus_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)].index)
        if processed_assoc_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)].index)
        indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['source'] == merge_source_check].index)

        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)

        if reference_record['source'] == merge_source_check:
            if 'created_at_month' in associated_records_df.columns: associated_records_df.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not associated_records_df.empty: failed_prox_audit_records_list.append(associated_records_df.copy())
            continue

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        indices_to_remove.update(remaining_associated_records_df[(remaining_associated_records_df['source'] == ref_source) & (remaining_associated_records_df['created_at_month'] == ref_year_month)].index)
        removed_associated_records_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)]

        for df_target in [removed_associated_records_df, remaining_associated_records_df]:
            if 'created_at_month' in df_target.columns: df_target.drop(columns=['created_at_month'], inplace=True)
        if not removed_associated_records_df.empty: failed_prox_audit_records_list.append(removed_associated_records_df)

        if len(remaining_associated_records_df) > 0:
            case3_auto_merge_further_filter_list.append(pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False))
            for index, assoc_record in remaining_associated_records_df.iterrows():
                final_faa = determine_faa_study_number(ref_fcc, ref_faa, assoc_record['faa_study_number'])
                if pd.notnull(final_faa):
                    merged_record_df = pd.DataFrame([merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)], columns=cols)
                    new_prox_audit_records_list.append(merged_record_df)
                    case3_post_auto_merge_list.append(merged_record_df)
                    case3_raw_post_auto_merge_list.extend([merged_record_df, pd.DataFrame([assoc_record], columns=cols), ref_record_cleaned.copy()])
                    if pd.notnull(merged_record_df['focus_asset_id'].iloc[0]): processed_focus_ids.add(merged_record_df['focus_asset_id'].iloc[0])
                    if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
                else:
                    failed_prox_audit_records_list.extend([ref_record_cleaned, pd.DataFrame([assoc_record], columns=cols)])
        else: failed_prox_audit_records_list.append(ref_record_cleaned)

    if new_prox_audit_records_list: case3_prox_audits_post_auto_merge_table = pd.concat([case3_prox_audits_post_auto_merge_table] + new_prox_audit_records_list, ignore_index=True)
    if failed_prox_audit_records_list: case3_prox_audits_post_auto_merge_table = pd.concat([case3_prox_audits_post_auto_merge_table] + failed_prox_audit_records_list, ignore_index=True)

    case3_auto_merge_further_filter = pd.concat(case3_auto_merge_further_filter_list, ignore_index=False) if case3_auto_merge_further_filter_list else pd.DataFrame(columns=cols)
    case3_post_auto_merge_table = pd.concat(case3_post_auto_merge_list, ignore_index=True) if case3_post_auto_merge_list else pd.DataFrame(columns=cols)
    case3_raw_post_auto_merge_table = pd.concat(case3_raw_post_auto_merge_list, ignore_index=True) if case3_raw_post_auto_merge_list else pd.DataFrame(columns=cols)
    print(f"--- Case 3 Processing completed in: {(time.time() - start_time) / 60:.2f} minutes ---")

    return (
        case3_auto_merge_further_filter,
        case3_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case3_post_auto_merge_table.drop_duplicates(ignore_index=True),
        case3_raw_post_auto_merge_table.drop_duplicates(ignore_index=True)
    )

def apply_case_3_maintenance_logic(
    prox_audits_table: pd.DataFrame,
    post_auto_merge_table: pd.DataFrame,
    post_merge_table: pd.DataFrame,
    raw_post_merge_table: pd.DataFrame,
    running_final_asset_table: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    if post_auto_merge_table.empty:
        return running_final_asset_table, pd.DataFrame(columns=prox_audits_table.columns)

    working_post_merge = post_auto_merge_table.reset_index(drop=True)
    final_asset_table_list = []
    associated_records_mask = working_post_merge['associated_asset_id'].notnull()

    if not post_merge_table.empty:
        post_merge_lookup = post_merge_table.set_index('focus_asset_id')
        cols_to_update = [col for col in working_post_merge.columns.tolist() if col not in ["audit_reason", "distance_to_reference", "agldiff_to_reference", "associated_asset_id", "index"]]
        for idx, assoc_record in working_post_merge[associated_records_mask].iterrows():
            if assoc_record['associated_asset_id'] in post_merge_lookup.index:
                matching_merged_record = post_merge_lookup.loc[assoc_record['associated_asset_id']]
                if isinstance(matching_merged_record, pd.DataFrame): matching_merged_record = matching_merged_record.iloc[0]
                for col in cols_to_update:
                    if col in matching_merged_record.index and col in working_post_merge.columns:
                        working_post_merge.loc[idx, col] = matching_merged_record[col]

    if not raw_post_merge_table.empty:
        raw_assoc_ids = set(raw_post_merge_table['associated_asset_id'].dropna())
        removal_mask = (working_post_merge['associated_asset_id'].notnull()) & (working_post_merge['associated_asset_id'].isin(raw_assoc_ids))
        working_post_merge = working_post_merge[~removal_mask]

    valid_groups = working_post_merge['focus_asset_id'].dropna()
    if not valid_groups.empty:
        group_sizes = working_post_merge.groupby('focus_asset_id').size()
        single_record_groups = group_sizes[group_sizes == 1].index
        final_asset_table_list.append(working_post_merge[working_post_merge['focus_asset_id'].isin(single_record_groups)].copy())
        working_post_merge = working_post_merge[~working_post_merge['focus_asset_id'].isin(single_record_groups)]

    sorted_post_merge_table = working_post_merge.sort_values(by=['focus_asset_id', 'associated_asset_id'], ascending=[True, True], na_position='first').reset_index(drop=True)
    case3_final_assets = pd.concat(final_asset_table_list, ignore_index=True) if final_asset_table_list else pd.DataFrame(columns=prox_audits_table.columns)

    aggregated_final_asset_table = pd.concat([running_final_asset_table, case3_final_assets], ignore_index=True)
    return aggregated_final_asset_table, sorted_post_merge_table

# ----------------- CASE 4 -----------------
def split_case_4_audits(case3_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = clean_asr_in_dataframe(case3_sorted_post_merge_table.copy())
    candidates_indices = set()
    for focus_asset_id, group in working_table.groupby('focus_asset_id'):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        if len(reference_record_df) == 1:
            reference_record = reference_record_df.iloc[0]
            if pd.isnull(reference_record['fcc_asr_number']) and pd.isnull(reference_record['faa_study_number']):
                final_matching_mask = (associated_records_df['fcc_asr_number'].notnull()) & (associated_records_df['faa_study_number'].notnull()) & \
                                      (associated_records_df['operator_name'] == reference_record['operator_name']) & \
                                      (associated_records_df['operator_site_id'] == reference_record['operator_site_id']) & \
                                      (associated_records_df['asset_status'] == reference_record['asset_status']) & \
                                      (associated_records_df['type'] == reference_record['type']) & \
                                      (associated_records_df['name'] == reference_record['name'])
                matching_associated_records = associated_records_df[final_matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.update([reference_record_df.index[0]] + list(matching_associated_records.index))
    case4_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    case4_prox_audits_post_auto_merge_table = working_table.loc[list(set(working_table.index).difference(candidates_indices))].copy()
    cols = working_table.columns
    if case4_auto_merge_candidates.empty: case4_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case4_prox_audits_post_auto_merge_table.empty: case4_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case4_auto_merge_candidates, case4_prox_audits_post_auto_merge_table

def apply_case_4_full_processing(candidates_table: pd.DataFrame, initial_prox_audits_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    start_time = time.time()
    print("Starting Case 4 processing...")
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    case4_auto_merge_further_filter_list, case4_raw_post_auto_merge_list, case4_post_auto_merge_list = [], [], []
    new_prox_audit_records_list, failed_prox_audit_records_list = [], []
    case4_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    processed_focus_ids, processed_assoc_ids = set(), set()

    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']): candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')

    for focus_asset_id, group in tqdm(candidates_table.groupby('focus_asset_id'), desc="Processing Case 4"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        if len(reference_record_df) != 1: continue
        reference_record = reference_record_df.iloc[0]
        ref_source, ref_year_month = reference_record['source'], reference_record['created_at_month']
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        if processed_focus_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)].index)
        if processed_assoc_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)].index)
        indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['source'] == merge_source_check].index)

        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        if reference_record['source'] == merge_source_check:
            if 'created_at_month' in associated_records_df.columns: associated_records_df.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not associated_records_df.empty: failed_prox_audit_records_list.append(associated_records_df.copy())
            continue

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        indices_to_remove.update(remaining_associated_records_df[(remaining_associated_records_df['source'] == ref_source) & (remaining_associated_records_df['created_at_month'] == ref_year_month)].index)
        removed_after_abcde_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)].copy()

        indices_to_remove_f = set()
        if not remaining_associated_records_df.empty:
            ref_agl = pd.to_numeric(reference_record['agl'], errors='coerce')
            if pd.notnull(ref_agl):
                assoc_agl_series = pd.to_numeric(remaining_associated_records_df['agl'], errors='coerce')
                diff = ((ref_agl - assoc_agl_series).abs() / assoc_agl_series) * 100
                diff_filled = diff.fillna(0).replace([np.inf, -np.inf], 999)
                indices_to_remove_f.update(remaining_associated_records_df[diff_filled > 25].index)

        removed_after_f_df = remaining_associated_records_df.loc[list(indices_to_remove_f)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove_f)].copy()

        removed_after_g_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        if len(remaining_associated_records_df) > 1:
            remaining_associated_records_df['abs_agldiff'] = pd.to_numeric(remaining_associated_records_df['agldiff_to_reference'], errors='coerce').abs()
            if not remaining_associated_records_df['abs_agldiff'].isnull().all():
                closest_record_index = remaining_associated_records_df['abs_agldiff'].idxmin()
                closest_record_df = remaining_associated_records_df.loc[[closest_record_index]]
                removed_after_g_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin([closest_record_index])]
                remaining_associated_records_df = closest_record_df
            else:
                removed_after_g_df = remaining_associated_records_df.copy()
                remaining_associated_records_df = pd.DataFrame(columns=remaining_associated_records_df.columns)

        if 'abs_agldiff' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['abs_agldiff'], inplace=True)

        all_removed_associated_records_df = pd.concat([removed_after_abcde_df, removed_after_f_df, removed_after_g_df], ignore_index=False)
        for df_target in [all_removed_associated_records_df, remaining_associated_records_df]:
            if 'created_at_month' in df_target.columns: df_target.drop(columns=['created_at_month'], inplace=True)

        if not all_removed_associated_records_df.empty: failed_prox_audit_records_list.append(all_removed_associated_records_df)

        if len(remaining_associated_records_df) == 1:
            case4_auto_merge_further_filter_list.append(pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False))
            assoc_record = remaining_associated_records_df.iloc[0]
            final_faa = determine_faa_study_number(assoc_record['fcc_asr_number'], reference_record['faa_study_number'], assoc_record['faa_study_number'])
            if pd.notnull(final_faa):
                merged_record_df = pd.DataFrame([merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=final_faa)], columns=cols)
                new_prox_audit_records_list.append(merged_record_df)
                case4_post_auto_merge_list.append(merged_record_df)
                case4_raw_post_auto_merge_list.extend([merged_record_df, pd.DataFrame([assoc_record], columns=cols), ref_record_cleaned.copy()])
                if pd.notnull(merged_record_df['focus_asset_id'].iloc[0]): processed_focus_ids.add(merged_record_df['focus_asset_id'].iloc[0])
                if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
            else:
                failed_prox_audit_records_list.extend([ref_record_cleaned, pd.DataFrame([assoc_record], columns=cols)])
        else: failed_prox_audit_records_list.append(ref_record_cleaned)

    if new_prox_audit_records_list: case4_prox_audits_post_auto_merge_table = pd.concat([case4_prox_audits_post_auto_merge_table] + new_prox_audit_records_list, ignore_index=True)
    if failed_prox_audit_records_list: case4_prox_audits_post_auto_merge_table = pd.concat([case4_prox_audits_post_auto_merge_table] + failed_prox_audit_records_list, ignore_index=True)

    case4_auto_merge_further_filter = pd.concat(case4_auto_merge_further_filter_list, ignore_index=False) if case4_auto_merge_further_filter_list else pd.DataFrame(columns=cols)
    case4_post_auto_merge_table = pd.concat(case4_post_auto_merge_list, ignore_index=True) if case4_post_auto_merge_list else pd.DataFrame(columns=cols)
    case4_raw_post_auto_merge_table = pd.concat(case4_raw_post_auto_merge_list, ignore_index=True) if case4_raw_post_auto_merge_list else pd.DataFrame(columns=cols)
    print(f"--- Case 4 Processing completed in: {(time.time() - start_time) / 60:.2f} minutes ---")
    return (case4_auto_merge_further_filter, case4_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True), case4_post_auto_merge_table.drop_duplicates(ignore_index=True), case4_raw_post_auto_merge_table.drop_duplicates(ignore_index=True))

def apply_case_4_maintenance_logic(prox_audits_table, post_auto_merge_table, post_merge_table, raw_post_merge_table, running_final_asset_table):
    return apply_case_2_maintenance_logic(prox_audits_table, post_auto_merge_table, post_merge_table, raw_post_merge_table, running_final_asset_table)


# ----------------- CASE 5 -----------------
def split_case_5_audits(case4_sorted_post_merge_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    working_table = clean_asr_in_dataframe(case4_sorted_post_merge_table.copy())
    candidates_indices = set()
    for focus_asset_id, group in working_table.groupby('focus_asset_id'):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()]
        if len(reference_record_df) == 1:
            reference_record = reference_record_df.iloc[0]
            if pd.isnull(reference_record['fcc_asr_number']) and pd.isnull(reference_record['faa_study_number']):
                final_matching_mask = (associated_records_df['fcc_asr_number'].isnull()) & (associated_records_df['faa_study_number'].isnull()) & \
                                      (associated_records_df['operator_name'] == reference_record['operator_name']) & \
                                      (associated_records_df['operator_site_id'] == reference_record['operator_site_id']) & \
                                      (associated_records_df['asset_status'] == reference_record['asset_status']) & \
                                      (associated_records_df['type'] == reference_record['type']) & \
                                      (associated_records_df['name'] == reference_record['name'])
                matching_associated_records = associated_records_df[final_matching_mask]
                if not matching_associated_records.empty:
                    candidates_indices.update([reference_record_df.index[0]] + list(matching_associated_records.index))
    case5_auto_merge_candidates = working_table.loc[list(candidates_indices)].copy()
    case5_prox_audits_post_auto_merge_table = working_table.loc[list(set(working_table.index).difference(candidates_indices))].copy()
    cols = working_table.columns
    if case5_auto_merge_candidates.empty: case5_auto_merge_candidates = pd.DataFrame(columns=cols)
    if case5_prox_audits_post_auto_merge_table.empty: case5_prox_audits_post_auto_merge_table = pd.DataFrame(columns=cols)
    return case5_auto_merge_candidates, case5_prox_audits_post_auto_merge_table

def apply_case_5_full_processing(candidates_table: pd.DataFrame, initial_prox_audits_table: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    start_time = time.time()
    print("Starting Case 5 processing...")
    merging_timestamp = datetime.now()
    cols = candidates_table.columns
    merge_source_check = f"Auto-Merged {merging_timestamp.strftime('%m/%Y')}"
    case5_auto_merge_further_filter_list, case5_raw_post_auto_merge_list, case5_post_auto_merge_list = [], [], []
    new_prox_audit_records_list, failed_prox_audit_records_list = [], []
    case5_prox_audits_post_auto_merge_table = initial_prox_audits_table.copy()
    processed_focus_ids, processed_assoc_ids = set(), set()

    if not pd.api.types.is_datetime64_any_dtype(candidates_table['created_at']): candidates_table['created_at'] = pd.to_datetime(candidates_table['created_at'], errors='coerce')
    candidates_table['created_at_month'] = candidates_table['created_at'].dt.to_period('M')

    for focus_asset_id, group in tqdm(candidates_table.groupby('focus_asset_id'), desc="Processing Case 5"):
        reference_record_df = group[group['associated_asset_id'].isnull()]
        associated_records_df = group[group['associated_asset_id'].notnull()].copy()
        if len(reference_record_df) != 1: continue
        reference_record = reference_record_df.iloc[0]
        ref_source, ref_year_month = reference_record['source'], reference_record['created_at_month']
        remaining_associated_records_df = associated_records_df.copy()
        indices_to_remove = set()
        if processed_focus_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_focus_ids)].index)
        if processed_assoc_ids: indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['associated_asset_id'].isin(processed_assoc_ids)].index)
        indices_to_remove.update(remaining_associated_records_df[remaining_associated_records_df['source'] == merge_source_check].index)

        ref_record_cleaned = reference_record_df.copy()
        if 'created_at_month' in ref_record_cleaned.columns: ref_record_cleaned.drop(columns=['created_at_month'], inplace=True)
        if reference_record['source'] == merge_source_check:
            if 'created_at_month' in associated_records_df.columns: associated_records_df.drop(columns=['created_at_month'], inplace=True)
            failed_prox_audit_records_list.append(ref_record_cleaned)
            if not associated_records_df.empty: failed_prox_audit_records_list.append(associated_records_df.copy())
            continue

        remaining_associated_records_df['created_at_month'] = remaining_associated_records_df['created_at'].dt.to_period('M')
        indices_to_remove.update(remaining_associated_records_df[(remaining_associated_records_df['source'] == ref_source) & (remaining_associated_records_df['created_at_month'] == ref_year_month)].index)
        removed_after_abcde_df = remaining_associated_records_df.loc[list(indices_to_remove)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove)].copy()

        indices_to_remove_f = set()
        if not remaining_associated_records_df.empty:
            ref_agl = pd.to_numeric(reference_record['agl'], errors='coerce')
            if pd.notnull(ref_agl):
                assoc_agl_series = pd.to_numeric(remaining_associated_records_df['agl'], errors='coerce')
                diff = ((ref_agl - assoc_agl_series).abs() / assoc_agl_series) * 100
                diff_filled = diff.fillna(0).replace([np.inf, -np.inf], 999)
                indices_to_remove_f.update(remaining_associated_records_df[diff_filled > 25].index)

        removed_after_f_df = remaining_associated_records_df.loc[list(indices_to_remove_f)].copy()
        remaining_associated_records_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin(indices_to_remove_f)].copy()

        removed_after_g_df = pd.DataFrame(columns=remaining_associated_records_df.columns)
        if len(remaining_associated_records_df) > 1:
            remaining_associated_records_df['abs_agldiff'] = pd.to_numeric(remaining_associated_records_df['agldiff_to_reference'], errors='coerce').abs()
            if not remaining_associated_records_df['abs_agldiff'].isnull().all():
                closest_record_index = remaining_associated_records_df['abs_agldiff'].idxmin()
                closest_record_df = remaining_associated_records_df.loc[[closest_record_index]]
                removed_after_g_df = remaining_associated_records_df.loc[~remaining_associated_records_df.index.isin([closest_record_index])]
                remaining_associated_records_df = closest_record_df
            else:
                removed_after_g_df = remaining_associated_records_df.copy()
                remaining_associated_records_df = pd.DataFrame(columns=remaining_associated_records_df.columns)

        if 'abs_agldiff' in remaining_associated_records_df.columns: remaining_associated_records_df.drop(columns=['abs_agldiff'], inplace=True)

        all_removed_associated_records_df = pd.concat([removed_after_abcde_df, removed_after_f_df, removed_after_g_df], ignore_index=False)
        for df_target in [all_removed_associated_records_df, remaining_associated_records_df]:
            if 'created_at_month' in df_target.columns: df_target.drop(columns=['created_at_month'], inplace=True)

        if not all_removed_associated_records_df.empty: failed_prox_audit_records_list.append(all_removed_associated_records_df)

        if len(remaining_associated_records_df) == 1:
            case5_auto_merge_further_filter_list.append(pd.concat([ref_record_cleaned, remaining_associated_records_df], ignore_index=False))
            assoc_record = remaining_associated_records_df.iloc[0]

            merged_record_df = pd.DataFrame([merge_records(reference_record, assoc_record, merging_timestamp, faa_study_number=None)], columns=cols)
            new_prox_audit_records_list.append(merged_record_df)
            case5_post_auto_merge_list.append(merged_record_df)
            case5_raw_post_auto_merge_list.extend([merged_record_df, pd.DataFrame([assoc_record], columns=cols), ref_record_cleaned.copy()])
            if pd.notnull(merged_record_df['focus_asset_id'].iloc[0]): processed_focus_ids.add(merged_record_df['focus_asset_id'].iloc[0])
            if pd.notnull(assoc_record['associated_asset_id']): processed_assoc_ids.add(assoc_record['associated_asset_id'])
        else: failed_prox_audit_records_list.append(ref_record_cleaned)

    if new_prox_audit_records_list: case5_prox_audits_post_auto_merge_table = pd.concat([case5_prox_audits_post_auto_merge_table] + new_prox_audit_records_list, ignore_index=True)
    if failed_prox_audit_records_list: case5_prox_audits_post_auto_merge_table = pd.concat([case5_prox_audits_post_auto_merge_table] + failed_prox_audit_records_list, ignore_index=True)

    case5_auto_merge_further_filter = pd.concat(case5_auto_merge_further_filter_list, ignore_index=False) if case5_auto_merge_further_filter_list else pd.DataFrame(columns=cols)
    case5_post_auto_merge_table = pd.concat(case5_post_auto_merge_list, ignore_index=True) if case5_post_auto_merge_list else pd.DataFrame(columns=cols)
    case5_raw_post_auto_merge_table = pd.concat(case5_raw_post_auto_merge_list, ignore_index=True) if case5_raw_post_auto_merge_list else pd.DataFrame(columns=cols)
    print(f"--- Case 5 Processing completed in: {(time.time() - start_time) / 60:.2f} minutes ---")
    return (case5_auto_merge_further_filter, case5_prox_audits_post_auto_merge_table.drop_duplicates(ignore_index=True), case5_post_auto_merge_table.drop_duplicates(ignore_index=True), case5_raw_post_auto_merge_table.drop_duplicates(ignore_index=True))

def apply_case_5_maintenance_logic(prox_audits_table, post_auto_merge_table, post_merge_table, raw_post_merge_table, running_final_asset_table):
    return apply_case_2_maintenance_logic(prox_audits_table, post_auto_merge_table, post_merge_table, raw_post_merge_table, running_final_asset_table)


# ==============================================================================
# 4. EXECUTION PIPELINE
# ==============================================================================

# CASE 1
case1_auto_merge_candidates, initial_case1_prox_audits_post_auto_merge_table = split_case_1_audits(prox_audits_table)
case1_auto_merge_further_filter, updated_case1_prox_audits_post_auto_merge_table, case1_post_auto_merge_table, case1_raw_post_auto_merge_table = apply_case_1_full_processing(case1_auto_merge_candidates, initial_case1_prox_audits_post_auto_merge_table)
case1_aggregated_final_asset_table, final_case1_prox_audits_post_auto_merge_table = apply_case_1_maintenance_logic(prox_audits_table, updated_case1_prox_audits_post_auto_merge_table, case1_post_auto_merge_table, case1_raw_post_auto_merge_table)

case1_aggregated_final_asset_table.to_csv('case1_aggregated_final_asset_table.csv')
case1_auto_merge_candidates.to_csv('case1_auto_merge_candidates.csv')
case1_post_auto_merge_table.to_csv('case1_post_auto_merge_table.csv')
case1_raw_post_auto_merge_table.to_csv('case1_raw_post_auto_merge_table.csv')
final_case1_prox_audits_post_auto_merge_table.to_csv('final_case1_prox_audits_post_auto_merge_table.csv')
initial_case1_prox_audits_post_auto_merge_table.to_csv('initial_case1_prox_audits_post_auto_merge_table.csv')
updated_case1_prox_audits_post_auto_merge_table.to_csv('updated_case1_prox_audits_post_auto_merge_table.csv')
case1_auto_merge_further_filter.to_csv('case1_auto_merge_further_filter.csv')

# CASE 2
case2_auto_merge_candidates, initial_case2_prox_audits_post_auto_merge_table = split_case_2_audits(final_case1_prox_audits_post_auto_merge_table)
case2_auto_merge_further_filter, updated_case2_prox_audits_post_auto_merge_table, case2_post_auto_merge_table, case2_raw_post_auto_merge_table = apply_case_2_full_processing(case2_auto_merge_candidates, initial_case2_prox_audits_post_auto_merge_table)
case2_aggregated_final_asset_table, final_case2_prox_audits_post_auto_merge_table = apply_case_2_maintenance_logic(prox_audits_table, updated_case2_prox_audits_post_auto_merge_table, case2_post_auto_merge_table, case2_raw_post_auto_merge_table, case1_aggregated_final_asset_table)

case2_aggregated_final_asset_table.to_csv('case2_aggregated_final_asset_table.csv')
case2_auto_merge_candidates.to_csv('case2_auto_merge_candidates.csv')
case2_post_auto_merge_table.to_csv('case2_post_auto_merge_table.csv')
case2_raw_post_auto_merge_table.to_csv('case2_raw_post_auto_merge_table.csv')
final_case2_prox_audits_post_auto_merge_table.to_csv('final_case2_prox_audits_post_auto_merge_table.csv')
initial_case2_prox_audits_post_auto_merge_table.to_csv('initial_case2_prox_audits_post_auto_merge_table.csv')
updated_case2_prox_audits_post_auto_merge_table.to_csv('updated_case2_prox_audits_post_auto_merge_table.csv')
case2_auto_merge_further_filter.to_csv('case2_auto_merge_further_filter.csv')

# CASE 3
case3_auto_merge_candidates, initial_case3_prox_audits_post_auto_merge_table = split_case_3_audits(final_case2_prox_audits_post_auto_merge_table)
case3_auto_merge_further_filter, updated_case3_prox_audits_post_auto_merge_table, case3_post_auto_merge_table, case3_raw_post_auto_merge_table = apply_case_3_full_processing(case3_auto_merge_candidates, initial_case3_prox_audits_post_auto_merge_table)
case3_aggregated_final_asset_table, final_case3_prox_audits_post_auto_merge_table = apply_case_3_maintenance_logic(prox_audits_table, updated_case3_prox_audits_post_auto_merge_table, case3_post_auto_merge_table, case3_raw_post_auto_merge_table, case2_aggregated_final_asset_table)

case3_aggregated_final_asset_table.to_csv('case3_aggregated_final_asset_table.csv')
case3_auto_merge_candidates.to_csv('case3_auto_merge_candidates.csv')
case3_post_auto_merge_table.to_csv('case3_post_auto_merge_table.csv')
case3_raw_post_auto_merge_table.to_csv('case3_raw_post_auto_merge_table.csv')
final_case3_prox_audits_post_auto_merge_table.to_csv('final_case3_prox_audits_post_auto_merge_table.csv')
initial_case3_prox_audits_post_auto_merge_table.to_csv('initial_case3_prox_audits_post_auto_merge_table.csv')
updated_case3_prox_audits_post_auto_merge_table.to_csv('updated_case3_prox_audits_post_auto_merge_table.csv')
case3_auto_merge_further_filter.to_csv('case3_auto_merge_further_filter.csv')

# CASE 4
case4_auto_merge_candidates, initial_case4_prox_audits_post_auto_merge_table = split_case_4_audits(final_case3_prox_audits_post_auto_merge_table)
case4_auto_merge_further_filter, updated_case4_prox_audits_post_auto_merge_table, case4_post_auto_merge_table, case4_raw_post_auto_merge_table = apply_case_4_full_processing(case4_auto_merge_candidates, initial_case4_prox_audits_post_auto_merge_table)
case4_aggregated_final_asset_table, final_case4_prox_audits_post_auto_merge_table = apply_case_4_maintenance_logic(prox_audits_table, updated_case4_prox_audits_post_auto_merge_table, case4_post_auto_merge_table, case4_raw_post_auto_merge_table, case3_aggregated_final_asset_table)

case4_aggregated_final_asset_table.to_csv('case4_aggregated_final_asset_table.csv')
case4_auto_merge_candidates.to_csv('case4_auto_merge_candidates.csv')
case4_post_auto_merge_table.to_csv('case4_post_auto_merge_table.csv')
case4_raw_post_auto_merge_table.to_csv('case4_raw_post_auto_merge_table.csv')
final_case4_prox_audits_post_auto_merge_table.to_csv('final_case4_prox_audits_post_auto_merge_table.csv')
initial_case4_prox_audits_post_auto_merge_table.to_csv('initial_case4_prox_audits_post_auto_merge_table.csv')
updated_case4_prox_audits_post_auto_merge_table.to_csv('updated_case4_prox_audits_post_auto_merge_table.csv')
case4_auto_merge_further_filter.to_csv('case4_auto_merge_further_filter.csv')

# CASE 5
case5_auto_merge_candidates, initial_case5_prox_audits_post_auto_merge_table = split_case_5_audits(final_case4_prox_audits_post_auto_merge_table)
case5_auto_merge_further_filter, updated_case5_prox_audits_post_auto_merge_table, case5_post_auto_merge_table, case5_raw_post_auto_merge_table = apply_case_5_full_processing(case5_auto_merge_candidates, initial_case5_prox_audits_post_auto_merge_table)
case5_aggregated_final_asset_table, final_case5_prox_audits_post_auto_merge_table = apply_case_5_maintenance_logic(prox_audits_table, updated_case5_prox_audits_post_auto_merge_table, case5_post_auto_merge_table, case5_raw_post_auto_merge_table, case4_aggregated_final_asset_table)

case5_aggregated_final_asset_table.to_csv('case5_aggregated_final_asset_table.csv')
case5_auto_merge_candidates.to_csv('case5_auto_merge_candidates.csv')
case5_post_auto_merge_table.to_csv('case5_post_auto_merge_table.csv')
case5_raw_post_auto_merge_table.to_csv('case5_raw_post_auto_merge_table.csv')
final_case5_prox_audits_post_auto_merge_table.to_csv('final_case5_prox_audits_post_auto_merge_table.csv')
initial_case5_prox_audits_post_auto_merge_table.to_csv('initial_case5_prox_audits_post_auto_merge_table.csv')
updated_case5_prox_audits_post_auto_merge_table.to_csv('updated_case5_prox_audits_post_auto_merge_table.csv')
case5_auto_merge_further_filter.to_csv('case5_auto_merge_further_filter.csv')

Starting Case 1 processing...


Processing Case 1: 100%|██████████| 82/82 [00:01<00:00, 75.44it/s]


--- Case 1 Processing completed in: 0.02 minutes ---
Starting Case 2 processing...


Processing Case 2: 100%|██████████| 2/2 [00:00<00:00, 88.95it/s]

--- Case 2 Processing completed in: 0.00 minutes ---


...

Starting Case 4 processing...


Processing Case 4: 100%|██████████| 7841/7841 [02:13<00:00, 58.95it/s]


--- Case 4 Processing completed in: 2.54 minutes ---
Starting Case 5 processing...


Processing Case 5: 100%|██████████| 1511/1511 [00:20<00:00, 72.60it/s]


--- Case 5 Processing completed in: 0.39 minutes ---


In [6]:
final_case5_prox_audits_post_auto_merge_table

,focus_asset_id,focus_asset,associated_asset_id,source,created_at,updated_at,latitude,longitude,name,operator_site_id,...,cdbs_facility_id,region,address,construction_date,stealth,asset_status,audit_reason,distance_to_reference,agldiff_to_reference,abs_agldiff
0,942614,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:53:42.284115,2023-01-13 15:25:12.493704,33.94619,-118.28200,CA12409,160407,...,<NA>,<NA>,"FIGUEROA ST WL 75F S OF 99TH ST S/N , August F...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
1,942614,Phoenix Tower International,943927,Phoenix Tower International,2019-08-22 21:36:14.93064,2023-06-26 12:05:50.800659,33.94574,-118.28200,MAIN ST EL 185F N OF 110TH ST SF,161773,...,<NA>,<NA>,"MAIN ST EL 185F N OF 110TH ST SF , , CA 90003,...",NaN,No,Unconfirmed,Proximity Audit (48m),49.914635,0.0,NaN
2,942664,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:55:21.34445,2023-01-13 11:49:35.586868,34.03929,-118.19900,CA12465,160463,...,<NA>,<NA>,"1ST ST SL 2F E OF DACOTAH ST E/W , Los Angeles...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
3,942664,Phoenix Tower International,942203,Phoenix Tower International,2019-08-22 20:40:05.047291,2023-01-13 13:04:52.922687,34.03910,-118.19900,CA11987,159985,...,<NA>,<NA>,"1ST ST SL 53F W OF FRESNO ST WF , Los Angeles,...",NaN,No,Unconfirmed,Proximity Audit (21m),21.075388,0.0,NaN
4,942761,Phoenix Tower International,<NA>,Phoenix Tower International,2019-08-22 20:58:33.921034,2023-01-13 12:41:13.613007,33.98899,-118.31900,CA12570,160568,...,<NA>,<NA>,"SLAUSON AVE NL 42F E OF 3RD AVE EF , Los Angel...",NaN,No,Unconfirmed,Focus Asset,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47358,1161828,ASR,264885,Crown Castle,2015-04-20 20:26:56.595238,2017-07-28 23:55:23.576292,39.00210,-95.73923,SKYLINE PARK/BURNETT'S MOUND,877829,...,<NA>,<NA>,"3511 SW SKYLINE PARKWAY , TOPEKA, KS 66614, UN...",2007-01-12,No,Active,Proximity Audit (7m),7.276909,4.5,NaN
47359,1161829,ASR,<NA>,ASR,2025-04-01 15:38:18.798044,2025-04-09 11:29:48.458842,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35045, UNITED STATES",2024-11-26,No,Active,Focus Asset,NaN,NaN,NaN
47360,1161829,ASR,1136163,ASR,2024-01-23 15:24:06.649527,2024-01-23 09:31:32.771277,32.81861,-86.61714,<NA>,<NA>,...,<NA>,<NA>,"Off Logan Road , Clanton, AL 35046, UNITED STATES",2024-11-26,No,Active,Proximity Audit (0m),0.000000,-20.4,NaN
47361,1162574,ASR,<NA>,ASR,2025-07-24 21:02:34.966713,2025-07-28 09:49:51.734224,42.34739,-91.47869,<NA>,<NA>,...,<NA>,<NA>,"3048 Hwy 13 IA-5254 , Ryan, IA 52330, UNITED S...",NaN,No,ASR-Granted,Focus Asset,NaN,NaN,NaN
